# SignSage v2 — Kaggle training run

Runs `research/training/train_sign_to_text.py` (corrected post-352868924 config).
Read `research/training/READINESS.md` first. Requires a Kaggle GPU + `HF_TOKEN` secret.
Each step below is one cell; outputs are the run record. Resume after interruption with `--resume`.

In [ ]:
!pip install -q pose-format jiwer sacrebleu rouge-score sentencepiece huggingface_hub requests tqdm pandas

In [ ]:
!git clone https://github.com/therandomuser03/signsage.git /kaggle/working/signsage
import os
OUT = "/kaggle/working/s2t_v2"
os.makedirs(OUT, exist_ok=True)
print("repo + out ready")
print("repo + out ready (run the provisioning cell next)")


In [ ]:
# Dataset provisioning: resolve iSign_v1.1.csv from (1) an existing working
# copy, (2) an attached Kaggle dataset (set ISIGN_DATASET_SLUG if attached
# under a custom slug), or (3) a Hugging Face download. Fails early with a
# clear error when none is available -- training never starts without data.
import os
from pathlib import Path

def _resolve_csv():
    cands = [Path("{CSV_PATH}")]
    slug = os.environ.get("ISIGN_DATASET_SLUG", "").strip()
    if slug:
        cands.append(Path("/kaggle/input") / slug / "iSign_v1.1.csv")
    if Path("/kaggle/input").exists():
        cands += sorted(Path("/kaggle/input").glob("*/iSign_v1.1.csv"))
    for p in cands:
        if p.is_file():
            return p
    try:
        from kaggle_secrets import UserSecretsClient
        token = os.environ.get("HF_TOKEN") or UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as exc:
        raise RuntimeError("HF_TOKEN Kaggle Secret (or env) is required to download iSign from Hugging Face.") from exc
    if not token:
        raise RuntimeError("HF_TOKEN is empty: attach the CSV as a Kaggle dataset or set the HF_TOKEN secret.")
    from huggingface_hub import hf_hub_download
    dest = Path("/kaggle/working/isign")
    dest.mkdir(parents=True, exist_ok=True)
    return Path(hf_hub_download(repo_id="Exploration-Lab/iSign", filename="iSign_v1.1.csv",
                                repo_type="dataset", token=token, local_dir=str(dest)))

CSV_PATH = str(_resolve_csv())
print("iSign CSV:", CSV_PATH)


In [ ]:
!cd /kaggle/working/signsage && python research/training/audit_dataset.py --csv {CSV_PATH} --out /kaggle/working/s2t_v2/dataset_audit.json

In [ ]:
# Overfit gate only: aborts here (no GPU wasted) if plumbing is broken.
!cd /kaggle/working/signsage && python research/training/train_sign_to_text.py --csv {CSV_PATH} --pose-source remote --out /kaggle/working/s2t_v2 --stages tokenizer,overfit

In [ ]:
# Full run (graceful stop at --max-hours). RESUME ACROSS SESSIONS:
# /kaggle/working is ephemeral -- --resume alone does NOT survive a new
# session. After each session: Save Version (output) to keep $OUT, then in
# the next session attach that output as input and copy it back, e.g.:
#   !cp -r /kaggle/input/<prior-output-slug>/s2t_v2 /kaggle/working/s2t_v2
# then re-run this same command with --resume (checkpoints + tokenizer
# are reloaded from /kaggle/working/s2t_v2).
!cd /kaggle/working/signsage && python research/training/train_sign_to_text.py --csv {CSV_PATH} --pose-source remote --out /kaggle/working/s2t_v2 --epochs 12 --max-hours 8 --resume


In [ ]:
# Evaluate + export (runs automatically at the end of train, or standalone):
!cd /kaggle/working/signsage && python research/training/train_sign_to_text.py --csv {CSV_PATH} --pose-source remote --out /kaggle/working/s2t_v2 --stages evaluate,export
import json
rep = json.load(open("/kaggle/working/s2t_v2/evaluation_report.json"))
print("test:", rep["test_metrics"])
print("diversity:", rep["diversity"])
print("gate:", rep["gate"])

In [ ]:
# Verify the exported bundle loads through the REAL serving adapter (no notebook state).
import sys; sys.path.insert(0, "/kaggle/working/signsage/backend")
import numpy as np
from app.ml.adapters.sign_to_text_adapter import SignToTextAdapter
a = SignToTextAdapter.load("/kaggle/working/s2t_v2/signsage_deployment_bundle", device="cpu")
r = a.transcribe(np.random.default_rng(0).normal(size=(25, 543, 3)).astype(np.float32))
print("bundle loads + transcribes:", r.accepted, r.reason, repr(r.text[:80]))
print("NOTE: random-input output says nothing about quality; see evaluation_report.json.")